<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/03_qsofa_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This will:
# 1. prepare hourly GCS
# 2. merge GCS into vitals
# 3. compute hourly qSOFA
# 4. summarize qSOFA into stay-level features
# 5. build a pure rule-based clinical qSOFA model
# 6. build an interpretable logistic regression model using qSOFA features
# 7. evaluate both on 6h / 12h / 24h

##1) Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import linregress

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

##2) Check input tables

In [ ]:
print('vitals_complete columns:')
print(vitals_complete.columns)

print('\ngcs_wide columns:')
print(gcs_wide.columns)

print('\nlabels_ordered columns:')
print(labels_ordered.columns)

##3) Prepare hourly GCS from gcs_wide

This converts charttime_hour into ICU-relative integer hour so it matches vitals_complete.

In [ ]:
def prepare_gcs_hourly(gcs_wide, cohort):
    gcs = gcs_wide.copy()

    gcs['charttime_hour'] = pd.to_datetime(gcs['charttime_hour'])
    gcs = gcs.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
    gcs['intime'] = pd.to_datetime(gcs['intime'])

    gcs['hours_since_admit'] = (
        (gcs['charttime_hour'] - gcs['intime']).dt.total_seconds() / 3600
    )

    gcs = gcs[
        (gcs['hours_since_admit'] >= 0) &
        (gcs['hours_since_admit'] < 24)
    ].copy()

    gcs['hour'] = gcs['hours_since_admit'].astype(int)

    # If multiple GCS rows exist in the same hour, keep the last one
    gcs = gcs.sort_values(['stay_id', 'hour', 'charttime_hour'])
    gcs = gcs.groupby(['stay_id', 'hour'], as_index=False).last()

    return gcs

In [ ]:
gcs_hourly = prepare_gcs_hourly(gcs_wide, cohort)

print('gcs_hourly shape:', gcs_hourly.shape)
gcs_hourly.head()

##4) Build hourly qSOFA in vitals_qsofa

qSOFA criteria:

- respiratory rate >= 22
- systolic BP <= 100
- GCS < 15

In [ ]:
def build_vitals_qsofa(vitals_complete, gcs_hourly):
    vc = vitals_complete.copy()

    # Merge hourly GCS
    vc = vc.merge(
        gcs_hourly[['stay_id', 'hour', 'gcs_total']],
        on=['stay_id', 'hour'],
        how='left'
    )

    # qSOFA components
    vc['qsofa_rr'] = (vc['resp_rate'] >= 22).astype(int)
    vc['qsofa_sbp'] = (vc['abp_sys'] <= 100).astype(int)
    vc['qsofa_gcs'] = ((vc['gcs_total'] < 15) & vc['gcs_total'].notna()).astype(int)

    # Total score
    vc['qsofa_total'] = vc['qsofa_rr'] + vc['qsofa_sbp'] + vc['qsofa_gcs']
    vc['qsofa_ge2'] = (vc['qsofa_total'] >= 2).astype(int)

    return vc

In [ ]:
vitals_qsofa = build_vitals_qsofa(vitals_complete, gcs_hourly)

print('vitals_qsofa shape:', vitals_qsofa.shape)
vitals_qsofa[['stay_id', 'hour', 'resp_rate', 'abp_sys', 'gcs_total', 'qsofa_total', 'qsofa_ge2']].head(15)

##5) Build stay-level qSOFA feature table

This creates summarized qSOFA features for each ICU stay.

In [ ]:
def compute_qsofa_features(vitals_qsofa):
    records = []

    for stay_id, group in vitals_qsofa.groupby('stay_id'):
        group = group.sort_values('hour')

        q_vals = group['qsofa_total'].values.astype(float)
        h_vals = group['hour'].values.astype(float)

        mask = ~np.isnan(q_vals)
        slope = np.nan
        if mask.sum() >= 2:
            slope, *_ = linregress(h_vals[mask], q_vals[mask])

        records.append({
            'stay_id': stay_id,

            'qsofa_mean_24h': np.nanmean(q_vals) if mask.any() else np.nan,
            'qsofa_std_24h': np.nanstd(q_vals) if mask.any() else np.nan,
            'qsofa_min_24h': np.nanmin(q_vals) if mask.any() else np.nan,
            'qsofa_max_24h': np.nanmax(q_vals) if mask.any() else np.nan,
            'qsofa_first_24h': q_vals[0] if len(q_vals) else np.nan,
            'qsofa_last_24h': q_vals[-1] if len(q_vals) else np.nan,
            'qsofa_slope_24h': slope,

            'qsofa_hours_ge2': int((q_vals >= 2).sum()) if mask.any() else 0,
            'qsofa_any_ge2': int((q_vals >= 2).any()) if mask.any() else 0,

            'rr_qsofa_hours': int(group['qsofa_rr'].sum()),
            'sbp_qsofa_hours': int(group['qsofa_sbp'].sum()),
            'gcs_qsofa_hours': int(group['qsofa_gcs'].sum()),

            'gcs_missing_frac': group['gcs_total'].isna().mean(),
        })

    return pd.DataFrame(records)

In [ ]:
qsofa_features = compute_qsofa_features(vitals_qsofa)

print('qsofa_features shape:', qsofa_features.shape)
qsofa_features.head()


##6) Build full qsofa_dataset

Merge:

- stay_ids_order
- qSOFA features
- labels

In [ ]:
def build_qsofa_dataset(stay_ids_order, qsofa_features, labels_ordered):
    qsofa_dataset = (
        pd.DataFrame({'stay_id': stay_ids_order})
        .merge(qsofa_features, on='stay_id', how='left')
        .merge(labels_ordered, on='stay_id', how='left')
    )
    return qsofa_dataset

In [ ]:
qsofa_dataset = build_qsofa_dataset(stay_ids_order, qsofa_features, labels_ordered)

print('qsofa_dataset shape:', qsofa_dataset.shape)
print(qsofa_dataset.columns.tolist())
qsofa_dataset.head()

##7) Add pure clinical qSOFA rule predictions

This is the medically standard baseline:

positive if qSOFA reached 2 or more

In [ ]:
def add_rule_based_qsofa_predictions(qsofa_dataset):
    df = qsofa_dataset.copy()

    df['pred_rule_any_ge2'] = (df['qsofa_any_ge2'] == 1).astype(int)
    df['pred_rule_last_ge2'] = (df['qsofa_last_24h'] >= 2).fillna(False).astype(int)
    df['pred_rule_max_ge2'] = (df['qsofa_max_24h'] >= 2).fillna(False).astype(int)

    return df

In [ ]:
qsofa_dataset = add_rule_based_qsofa_predictions(qsofa_dataset)

qsofa_dataset[[
    'stay_id',
    'qsofa_max_24h',
    'qsofa_last_24h',
    'qsofa_any_ge2',
    'pred_rule_any_ge2',
    'pred_rule_last_ge2',
    'pred_rule_max_ge2'
]].head()

##8) Evaluation function for the pure rule-based clinical model

In [ ]:
def evaluate_rule_based_model(df, target_label, eligible_col, pred_col):
    sub = df[df[eligible_col].fillna(False)].copy()

    y_true = sub[target_label].fillna(0).astype(int).values
    y_pred = sub[pred_col].fillna(0).astype(int).values

    print('=' * 70)
    print(f'Target label : {target_label}')
    print(f'Rule used    : {pred_col}')
    print(f'Eligible n   : {len(y_true):,}')
    print(f'Positive rate: {y_true.mean():.4f}')
    print('=' * 70)

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print('Confusion Matrix:')
    print(f'TN={tn}  FP={fp}')
    print(f'FN={fn}  TP={tp}')

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    print(f'\nAccuracy    : {accuracy:.4f}')
    print(f'Sensitivity : {sensitivity:.4f}')
    print(f'Specificity : {specificity:.4f}')
    print(f'Precision   : {precision:.4f}')
    print(f'NPV         : {npv:.4f}')

    try:
        auroc = roc_auc_score(y_true, y_pred)
        auprc = average_precision_score(y_true, y_pred)
        print(f'AUROC       : {auroc:.4f}')
        print(f'AUPRC       : {auprc:.4f}')
    except Exception as e:
        print('Could not compute AUROC/AUPRC:', e)

    return {
        'n': len(y_true),
        'pos_rate': y_true.mean(),
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'npv': npv,
    }

##9) Run the pure clinical qSOFA baseline on all horizons

In [ ]:
rule_results = {}

for target_label, eligible_col in [
    ('y_6h', 'eligible_6h'),
    ('y_12h', 'eligible_12h'),
    ('y_24h', 'eligible_24h'),
]:
    result = evaluate_rule_based_model(
        qsofa_dataset,
        target_label=target_label,
        eligible_col=eligible_col,
        pred_col='pred_rule_any_ge2'
    )
    rule_results[target_label] = result

In [ ]:
print('\nSummary of pure qSOFA rule:')
for h, r in rule_results.items():
    print(
        f'{h:6s} | '
        f'n={r["n"]:,} | '
        f'acc={r["accuracy"]:.4f} | '
        f'sens={r["sensitivity"]:.4f} | '
        f'spec={r["specificity"]:.4f} | '
        f'prec={r["precision"]:.4f} | '
        f'npv={r["npv"]:.4f}'
    )

##10) Build an interpretable ML model from qSOFA features

This is more convincing than pure qSOFA alone, because it stays clinically interpretable but learns better from the qSOFA trajectory.

In [ ]:
def train_qsofa_logistic_model(qsofa_dataset, target_label='y_24h', eligible_col='eligible_24h'):
    feature_cols = [
        'qsofa_mean_24h',
        'qsofa_std_24h',
        'qsofa_min_24h',
        'qsofa_max_24h',
        'qsofa_first_24h',
        'qsofa_last_24h',
        'qsofa_slope_24h',
        'qsofa_hours_ge2',
        'qsofa_any_ge2',
        'rr_qsofa_hours',
        'sbp_qsofa_hours',
        'gcs_qsofa_hours',
        'gcs_missing_frac',
    ]

    df = qsofa_dataset.copy()

    elig = df[eligible_col].fillna(False).astype(bool).values
    y = df[target_label].fillna(0).astype(int).values
    X_tab = df[feature_cols].values.astype(np.float32)

    X_elig = X_tab[elig]
    y_elig = y[elig]

    print(f'Target: {target_label}')
    print(f'Eligible stays: {len(y_elig):,}')
    print(f'Positive rate : {y_elig.mean():.4f}')

    idx = np.arange(len(y_elig))

    idx_trainval, idx_test = train_test_split(
        idx, test_size=0.15, random_state=42, stratify=y_elig
    )
    idx_train, idx_val = train_test_split(
        idx_trainval,
        test_size=0.15 / 0.85,
        random_state=42,
        stratify=y_elig[idx_trainval]
    )

    X_train, X_val, X_test = X_elig[idx_train], X_elig[idx_val], X_elig[idx_test]
    y_train, y_val, y_test = y_elig[idx_train], y_elig[idx_val], y_elig[idx_test]

    print(f'Train: {len(y_train):,} | positives: {y_train.sum():,} | rate: {y_train.mean():.4f}')
    print(f'Val  : {len(y_val):,} | positives: {y_val.sum():,} | rate: {y_val.mean():.4f}')
    print(f'Test : {len(y_test):,} | positives: {y_test.sum():,} | rate: {y_test.mean():.4f}')

    imputer = SimpleImputer(strategy='median')
    X_train = imputer.fit_transform(X_train)
    X_val = imputer.transform(X_val)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    model = LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_prob_val = model.predict_proba(X_val)[:, 1]
    y_prob_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = (y_prob_test >= 0.5).astype(int)

    val_auroc = roc_auc_score(y_val, y_prob_val)
    val_auprc = average_precision_score(y_val, y_prob_val)
    test_auroc = roc_auc_score(y_test, y_prob_test)
    test_auprc = average_precision_score(y_test, y_prob_test)

    print('\n' + '=' * 60)
    print(f'Validation AUROC: {val_auroc:.4f}')
    print(f'Validation AUPRC: {val_auprc:.4f}')
    print(f'Test AUROC      : {test_auroc:.4f}')
    print(f'Test AUPRC      : {test_auprc:.4f}')
    print('=' * 60)

    print('\nClassification Report (threshold = 0.5)')
    print(classification_report(y_test, y_pred_test, digits=4))

    cm = confusion_matrix(y_test, y_pred_test)
    tn, fp, fn, tp = cm.ravel()

    print('Confusion Matrix:')
    print(f'TN={tn}  FP={fp}')
    print(f'FN={fn}  TP={tp}')

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    print(f'Sensitivity: {sensitivity:.4f}')
    print(f'Specificity: {specificity:.4f}')
    print(f'Precision  : {precision:.4f}')
    print(f'NPV        : {npv:.4f}')

    return {
        'model': model,
        'imputer': imputer,
        'scaler': scaler,
        'feature_cols': feature_cols,
        'y_test': y_test,
        'y_prob_test': y_prob_test,
        'y_pred_test': y_pred_test,
        'val_auroc': val_auroc,
        'val_auprc': val_auprc,
        'test_auroc': test_auroc,
        'test_auprc': test_auprc,
    }

##11) Train logistic qSOFA model on 24h

In [ ]:
logreg_24h = train_qsofa_logistic_model(
    qsofa_dataset,
    target_label='y_24h',
    eligible_col='eligible_24h'
)

##12) Compare logistic qSOFA across all horizons

In [ ]:
logreg_results = {}

for target_label, eligible_col in [
    ('y_6h', 'eligible_6h'),
    ('y_12h', 'eligible_12h'),
    ('y_24h', 'eligible_24h'),
]:
    print('\n' + '=' * 70)
    print(f'Logistic qSOFA model for {target_label}')
    print('=' * 70)

    res = train_qsofa_logistic_model(
        qsofa_dataset,
        target_label=target_label,
        eligible_col=eligible_col
    )

    logreg_results[target_label] = {
        'val_auroc': res['val_auroc'],
        'val_auprc': res['val_auprc'],
        'test_auroc': res['test_auroc'],
        'test_auprc': res['test_auprc'],
    }

In [ ]:
print('\nSummary of logistic qSOFA model:')
for h, r in logreg_results.items():
    print(
        f'{h:6s} | '
        f'Val AUROC={r["val_auroc"]:.4f} | '
        f'Val AUPRC={r["val_auprc"]:.4f} | '
        f'Test AUROC={r["test_auroc"]:.4f} | '
        f'Test AUPRC={r["test_auprc"]:.4f}'
    )

##13) Plot ROC and PR curves for the logistic model

In [ ]:
def plot_qsofa_logistic_curves(y_test, y_prob, target_label='y_24h'):
    auroc = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, lw=2, label=f'AUROC = {auroc:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'ROC Curve — {target_label}')
    axes[0].legend()
    axes[0].grid(True)

    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    baseline = y_test.mean()
    axes[1].plot(rec, prec, lw=2, label=f'AUPRC = {auprc:.3f}')
    axes[1].axhline(y=baseline, color='k', linestyle='--', lw=1, label=f'Baseline = {baseline:.3f}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'PR Curve — {target_label}')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_qsofa_logistic_curves(
    logreg_24h['y_test'],
    logreg_24h['y_prob_test'],
    target_label='y_24h'
)

##14) Inspect which qSOFA features mattered most

In [ ]:
coef_df = pd.DataFrame({
    'feature': logreg_24h['feature_cols'],
    'coefficient': logreg_24h['model'].coef_[0]
}).sort_values('coefficient', key=np.abs, ascending=False)

coef_df

#What we now have

we now have two medically sensible models from scratch:

##Model 1 — Pure clinical baseline
uses qSOFA >= 2
easy to explain to instructor
clinically interpretable
medically applicable
##Model 2 — Interpretable ML baseline
logistic regression using qSOFA-derived features
still explainable
usually stronger than fixed bedside rule
more convincing academically